# FDTD Cloud Server — Colab/Kaggle Quickstart

**Steps:**
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Run all cells
3. Copy the `wss://...` URL printed in the last cell
4. Paste it into the **Remote Server URL** field in your browser app

The server stays alive as long as this notebook is running (up to ~12h on Colab free tier).

In [ ]:
# Install deps
!pip install -q 'fastapi[standard]' 'uvicorn[standard]' numpy

# Try CuPy for GPU acceleration (optional, may take a minute)
try:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'cupy-cuda12x'])
    print('CuPy installed — GPU acceleration enabled')
except Exception:
    print('CuPy not available — using NumPy (CPU)')

# Install cloudflared — /tmp always exists; /content only exists inside Colab
import urllib.request, os, stat
_cf_path = '/tmp/cloudflared'
print('Downloading cloudflared...')
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    _cf_path
)
os.chmod(_cf_path, os.stat(_cf_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print(f'cloudflared ready at {_cf_path}')

In [ ]:
# Write the server files
# (copy-pasted from cloud/fdtd_engine.py and cloud/fdtd_server.py)
import urllib.request, os

# If you have the repo cloned locally and pushed to GitHub, replace the URLs below.
# Otherwise, upload fdtd_engine.py and fdtd_server.py via the Colab sidebar.

print('Upload fdtd_engine.py and fdtd_server.py using the Files panel on the left,')
print('or run: from google.colab import files; files.upload()')

In [ ]:
# Alternative: upload files interactively
try:
    from google.colab import files
    uploaded = files.upload()  # select fdtd_engine.py and fdtd_server.py
except ImportError:
    print('Not in Colab — make sure fdtd_engine.py and fdtd_server.py are in the current directory')

In [ ]:
import threading, subprocess, time

server_proc = None

def start_server():
    global server_proc
    server_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'fdtd_server:app', '--port', '8765', '--host', '0.0.0.0'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in server_proc.stdout:
        print(line.decode().strip())

t = threading.Thread(target=start_server, daemon=True)
t.start()
time.sleep(3)
print('Server started on port 8765')

In [ ]:
import subprocess, threading, re, time

tunnel_url = None

def start_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8765'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in proc.stdout:
        decoded = line.decode().strip()
        match = re.search(r'https://[\w\-]+\.trycloudflare\.com', decoded)
        if match and tunnel_url is None:
            tunnel_url = match.group(0)
            print(f'\n=== Tunnel ready ===')
            ws_url = tunnel_url.replace('https://', 'wss://') + '/ws'
            print(f'WebSocket URL: {ws_url}')
            print(f'Paste this into the Remote Server URL field in your browser app.')
            print(f'====================')

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()

# Wait up to 30s for the URL
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)

if not tunnel_url:
    print('Tunnel URL not found yet — check the output above')

In [ ]:
# Keep the notebook alive (Colab disconnects idle kernels after ~90 min)
# Run this cell, then you can minimize the browser tab.
import time
print('Keeping alive... (interrupt kernel to stop)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)